# Fine-Tuning Orchestrator

Notebook ini mengatur pelatihan model Bi-Encoder dan Cross-Encoder untuk CV Matching System.

## Struktur Direktori
- `data/training/`: Dataset CSV (Triplet & Pairs)
- `training/scripts/`: Script pelatihan Python
- `models/`: Output model terlatih

## Langkah Persiapan
1. Pastikan dataset CSV tersedia di `data/training/`
2. Install dependencies: `pip install -r backend/requirements.txt`

In [6]:
# Setup Paths
import os
import sys

# Add training scripts to path
sys.path.append('training/scripts')

# Define paths
DATA_DIR = '/home/kuliah/intern/dicoding/caps-final/data/training'
MODEL_DIR = '/home/kuliah/intern/dicoding/caps-final/models'
SCRIPT_DIR = '/home/kuliah/intern/dicoding/caps-final/training/scripts'
SKILLS_DIR = '/home/kuliah/intern/dicoding/caps-final/backend/app/core/skills'
DATASET_SCRIPT = '/home/kuliah/intern/dicoding/caps-final/training/scripts/generate_dataset.py'
DATASET_TEMPLATE = '/home/kuliah/intern/dicoding/caps-final/training/templates'

# Check directory structure
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Data directory: {os.path.abspath(DATA_DIR)}")
print(f"Model directory: {os.path.abspath(MODEL_DIR)}")
print(f"Script directory: {os.path.abspath(SCRIPT_DIR)}")

Data directory: /home/kuliah/intern/dicoding/caps-final/data/training
Model directory: /home/kuliah/intern/dicoding/caps-final/models
Script directory: /home/kuliah/intern/dicoding/caps-final/training/scripts


## Phase 1: Data Preparation

Dataset yang diperlukan:
1. `bi_encoder_train.csv` (Triplet: anchor, positive, negative)
2. `cross_encoder_train.csv` (Pairs: cv_text, jd_text, label)

Anda dapat menggunakan data sintetis untuk testing atau siapkan data real dari CV dan JD Anda.

In [7]:
# Generate synthetic data using script
import subprocess

def generate_synthetic_data(num_triplets=2000, num_pairs=2000):
    print("Generating dataset using training/scripts/generate_dataset.py...")
    cmd = [
        'python', DATASET_SCRIPT,
        '--skills_dir', SKILLS_DIR,
        '--templates_dir', DATASET_TEMPLATE,
        '--output_dir', DATA_DIR,
        '--num_triplets', str(num_triplets),
        '--num_pairs', str(num_pairs)
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(f"Error: {result.stderr}")
    except Exception as e:
        print(f"Exception during generation: {e}")

# Run synthetic data generation
generate_synthetic_data()

# Check existing data
if os.path.exists(f'{DATA_DIR}/bi_encoder_train.csv'):
    print("Bi-Encoder training data found.")
else:
    print("Bi-Encoder training data NOT found.")

if os.path.exists(f'{DATA_DIR}/cross_encoder_train.csv'):
    print("Cross-Encoder training data found.")
else:
    print("Cross-Encoder training data NOT found.")

Generating dataset using training/scripts/generate_dataset.py...
Generated and saved Bi-Encoder triplet dataset to: /home/kuliah/intern/dicoding/caps-final/data/training/bi_encoder_train.csv
Generated and saved Cross-Encoder pair dataset to: /home/kuliah/intern/dicoding/caps-final/data/training/cross_encoder_train.csv

Bi-Encoder training data found.
Cross-Encoder training data found.


## Phase 2: Bi-Encoder Training

Melatih model Bi-Encoder menggunakan data triplet untuk menghasilkan embedding yang akurat.

In [8]:
import subprocess

def train_bi_encoder():
    train_csv = f'{DATA_DIR}/bi_encoder_train.csv'
    output_path = f'{MODEL_DIR}/bi-encoder-cv-matcher'
    
    if not os.path.exists(train_csv):
        print(f"Error: Training data not found at {train_csv}")
        return
    
    print(f"Training Bi-Encoder...")
    print(f"Input: {train_csv}")
    print(f"Output: {output_path}")
    
    # Run training script
    cmd = [
        'python', f'{SCRIPT_DIR}/train_bi_encoder.py',
        '--train_csv', train_csv,
        '--output_path', output_path,
        '--epochs', '5',
        '--batch_size', '16'
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(f"Error: {result.stderr}")
        else:
            print("Bi-Encoder training completed successfully!")
    except Exception as e:
        print(f"Exception during training: {e}")

# Uncomment to run training
train_bi_encoder()


Training Bi-Encoder...
Input: /home/kuliah/intern/dicoding/caps-final/data/training/bi_encoder_train.csv
Output: /home/kuliah/intern/dicoding/caps-final/models/bi-encoder-cv-matcher
Loading base model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Loading training data from: /home/kuliah/intern/dicoding/caps-final/data/training/bi_encoder_train.csv
Setting up training configurations...
Starting Bi-Encoder training...
{'loss': 3.7115, 'grad_norm': 46.931636810302734, 'learning_rate': 1.9712000000000004e-05, 'epoch': 0.08}
{'loss': 2.9179, 'grad_norm': 25.889936447143555, 'learning_rate': 1.9392000000000003e-05, 'epoch': 0.16}
{'loss': 2.5358, 'grad_norm': 22.375288009643555, 'learning_rate': 1.9072000000000003e-05, 'epoch': 0.24}
{'loss': 2.4798, 'grad_norm': 17.079326629638672, 'learning_rate': 1.8752000000000003e-05, 'epoch': 0.32}
{'loss': 2.0118, 'grad_norm': 14.356405258178711, 'learning_rate': 1.8432000000000002e-05, 'epoch': 0.4}
{'loss': 1.7442, 'grad_norm': 15.484

## Phase 3: Cross-Encoder Training

Melatih model Cross-Encoder untuk re-ranking kandidat dengan akurasi tinggi.

In [9]:
def train_cross_encoder():
    train_csv = f'{DATA_DIR}/cross_encoder_train.csv'
    output_path = f'{MODEL_DIR}/cross-encoder-cv-matcher'
    
    if not os.path.exists(train_csv):
        print(f"Error: Training data not found at {train_csv}")
        return
    
    print(f"Training Cross-Encoder...")
    print(f"Input: {train_csv}")
    print(f"Output: {output_path}")
    
    # Run training script
    cmd = [
        'python', f'{SCRIPT_DIR}/train_cross_encoder.py',
        '--train_csv', train_csv,
        '--output_path', output_path,
        '--epochs', '3',
        '--batch_size', '16'
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(f"Error: {result.stderr}")
        else:
            print("Cross-Encoder training completed successfully!")
    except Exception as e:
        print(f"Exception during training: {e}")

# Uncomment to run training
train_cross_encoder()


Training Cross-Encoder...
Input: /home/kuliah/intern/dicoding/caps-final/data/training/cross_encoder_train.csv
Output: /home/kuliah/intern/dicoding/caps-final/models/cross-encoder-cv-matcher
Loading base Cross-Encoder: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Loading training data from: /home/kuliah/intern/dicoding/caps-final/data/training/cross_encoder_train.csv
Starting Cross-Encoder training...
Saving Cross-Encoder model to /home/kuliah/intern/dicoding/caps-final/models/cross-encoder-cv-matcher...
Cross-Encoder training complete! Model saved to /home/kuliah/intern/dicoding/caps-final/models/cross-encoder-cv-matcher

Cross-Encoder training completed successfully!


## Phase 4: Model Verification

Verifikasi model yang telah dilatih tersimpan dengan benar.

In [10]:
from sentence_transformers import SentenceTransformer, CrossEncoder

def verify_models():
    # Check Bi-Encoder
    bi_encoder_path = f'{MODEL_DIR}/bi-encoder-cv-matcher'
    if os.path.exists(bi_encoder_path):
        try:
            model = SentenceTransformer(bi_encoder_path)
            print(f"✓ Bi-Encoder loaded successfully from {bi_encoder_path}")
            # Test inference
            test_embedding = model.encode("Test sentence")
            print(f"  Embedding shape: {test_embedding.shape}")
        except Exception as e:
            print(f"✗ Error loading Bi-Encoder: {e}")
    else:
        print(f"✗ Bi-Encoder model not found at {bi_encoder_path}")
    
    # Check Cross-Encoder
    cross_encoder_path = f'{MODEL_DIR}/cross-encoder-cv-matcher'
    if os.path.exists(cross_encoder_path):
        try:
            model = CrossEncoder(cross_encoder_path)
            print(f"✓ Cross-Encoder loaded successfully from {cross_encoder_path}")
            # Test inference
            test_score = model.predict([("Test CV", "Test JD")])
            print(f"  Test score: {test_score}")
        except Exception as e:
            print(f"✗ Error loading Cross-Encoder: {e}")
    else:
        print(f"✗ Cross-Encoder model not found at {cross_encoder_path}")

verify_models()


/home/alwand/.conda/envs/cv_project_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Bi-Encoder loaded successfully from /home/kuliah/intern/dicoding/caps-final/models/bi-encoder-cv-matcher
  Embedding shape: (384,)
✓ Cross-Encoder loaded successfully from /home/kuliah/intern/dicoding/caps-final/models/cross-encoder-cv-matcher
  Test score: [0.7071966]
